# Newly supported equation families

A short, executable tour of symbolic-coefficient polynomials, exact inequalities, multiple radicals, and trigonometric reductions.

In [1]:
import math
import sys
from pathlib import Path

# Use this source checkout when the notebook is run from the repository.
project_root = Path.cwd()
if not (project_root / 'kiwicalc').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import kiwicalc as kw

def readable(solution_set):
    if isinstance(solution_set, kw.UnionSolutionSet):
        return '\n    OR '.join(readable(part) for part in solution_set.sets)
    if isinstance(solution_set, kw.ConditionalSolutionSet):
        conditions = ' and '.join(solution_set.conditions)
        return f"{readable(solution_set.solution_set)}  if {conditions}"
    return str(solution_set)

def show(label, result):
    print(f"{label}: {readable(result.solution_set)}")
    print(f"  status={result.status}, exact={result.exact}, complete={result.complete}")

## 1. Symbolic-coefficient polynomials
The solver retains the conditions under which quadratic, linear, and repeated-root branches apply. It also recognizes useful low-degree structures such as biquadratics.

In [2]:
quadratic = kw.solve_equation('a*x^2 + b*x + c = 0', variable='x')
factored_low_degree = kw.solve_equation(
    '(x - 1)*(x + 2)*(a*x + b) = 0', variable='x'
)
show('symbolic quadratic', quadratic)
show('factored symbolic cubic', factored_low_degree)

symbolic quadratic: {(2*a)^-1*sqrt(-4*a*c + b^2) - 1*(2*a)^-1*b, -1*(2*a)^-1*b - 1*(2*a)^-1*sqrt(-4*a*c + b^2)}  if a != 0 and -4*a*c + b^2 > 0
    OR {-1*(2*a)^-1*b}  if a != 0 and -4*a*c + b^2 = 0
    OR {-1*b^-1*c}  if a = 0 and b != 0
    OR Reals  if a = 0 and b = 0 and c = 0
  status=solved, exact=True, complete=True
factored symbolic cubic: {-2, 1}
    OR {-1*a^-1*b}  if a != 0
    OR Reals  if a = 0 and b = 0
  status=solved, exact=True, complete=True


## 2. Exact polynomial and rational inequalities
Endpoints, open or closed boundaries, poles, and removable holes are represented exactly.

In [3]:
polynomial_inequality = kw.solve_inequality('x^2 - 5*x + 6 >= 0')
rational_inequality = kw.solve_inequality('(x^2 - 1)/(x - 1) > 0')
show('polynomial inequality', polynomial_inequality)
show('rational inequality', rational_inequality)

polynomial inequality: (-inf, 2]
    OR [3, inf)
  status=solved, exact=True, complete=True
rational inequality: (-1, 1)
    OR (1, inf)
  status=solved, exact=True, complete=True


## 3. Multiple and nested square roots
Radicals are isolated one at a time. Every candidate is checked against the original equation, so roots introduced by squaring are discarded.

In [4]:
two_radicals = kw.solve_equation('sqrt(x + 1) + sqrt(x - 1) = 3', variable='x')
nested_radical = kw.solve_equation('sqrt(x + sqrt(x)) = 2', variable='x')
show('two radicals', two_radicals)
show('nested radical', nested_radical)

two radicals: {85/36}
  status=solved, exact=True, complete=True
nested radical: {9/2 - 1/2*sqrt(17)}
  status=solved, exact=True, complete=True


## 4. Trigonometric reductions
The engine reduces supported double- and triple-angle forms and returns every solution inside a requested interval.

In [5]:
double_angle = kw.solve_equation(
    'sin(2*x) = cos(x)', variable='x', interval=(-2*math.pi, 2*math.pi)
)
triple_angle = kw.solve_equation(
    'cos(3*x) = cos(x)', variable='x', interval=(-2*math.pi, 2*math.pi)
)
show('double-angle reduction', double_angle)
show('triple-angle reduction', triple_angle)

double-angle reduction: {-11/6*pi, -3/2*pi, -7/6*pi, -1/2*pi, 1/6*pi, 1/2*pi, 5/6*pi, 3/2*pi}
  status=solved, exact=True, complete=True
triple-angle reduction: {-2*pi, -3/2*pi, -1*pi, -1/2*pi, 0, 1/2*pi, pi, 3/2*pi, 2*pi}
  status=solved, exact=True, complete=True


## Verification
These assertions make the notebook double as a quick smoke test.

In [6]:
results = [
    quadratic, factored_low_degree, polynomial_inequality, rational_inequality,
    two_radicals, nested_radical, double_angle, triple_angle,
]
assert all(result.status == 'solved' for result in results)
assert all(result.exact and result.complete for result in results)
print('All examples solved exactly and completely.')

All examples solved exactly and completely.
